# 08 - Retrain the answer model: all 12,825 training rows, re-ranked references, both GPUs

> Run with **Save Version -> Save & Run All (Commit)**. Settings: **Accelerator = GPU T4 x2**, **Internet = On**.
> One commit run is limited to 12 h. Training stops cleanly after `TRAIN_HOURS` and saves a checkpoint; if it has not finished,
> run the notebook **again** with this notebook's own output attached (see the end) and it **resumes** from that checkpoint.

**Inputs (Add Input):**
1. `train_processed.csv` (dataset `train-processed1`)
2. `question_neighbours_reranked.json.gz` (Your Work -> the re-ranker notebook 07 output)
3. *second session only:* this notebook's previous output (Your Work -> this notebook)

**What changed vs Phase 4 (and why)**

| | Phase 4 | this run | evidence |
|---|---|---|---|
| training rows | 4,000 random, 1 epoch | **all 12,825**, 1 epoch | loss still ~1.45; fine-tuning on little data doubled repetition loops |
| reference answers in the prompt | retrieval top-2 | **re-ranked top-2** (cross-fitted for train rows) | generated quality follows reference quality (r = 0.63); re-ranking +0.015 top-1 ROUGE-L on 1,500 hold-out rows |
| max length | 2,560 | **3,072** | 266/4,000 targets were cut and never showed an ending |
| GPUs | one adapter per GPU | **both GPUs, one model (DDP)** | 802 steps x ~83 s on one T4 = ~18 h |

Unchanged: Qwen2.5-3B-Instruct, 4-bit NF4, LoRA r=16/alpha=32 on all linear layers, lr 2e-4 cosine, effective batch 16,
loss on answer tokens only, 20% RAFT-style distractor references, the same system prompt.

In [ ]:
!pip install -q -U peft bitsandbytes accelerate huggingface_hub

In [ ]:
%%writefile prompts.py
"""Phase 4: build the training / inference prompts.

Modes
  none : question only                                   (baseline B2: plain fine-tuning)
  nn   : question + the 2 nearest training Q&A pairs      (proposed model without PDFs)
  full : nn + source-page excerpts                         (proposed model with PDFs)

Neighbours for TRAIN questions already exclude the question's own group (see Phase 2), so the model is trained to
synthesise an answer from related-but-different answers, not to copy an identical one.
Pure ASCII source on purpose.
"""
import json
import random

SYSTEM = (
    "You are an expert Indian agricultural advisor. Answer the farmer's question in English with a detailed, "
    "well-structured advisory: key facts or symptoms, thresholds where relevant, cultural, biological and chemical "
    "management with exact doses and units, and safety notes. Use the reference material when it applies to this "
    "question; never invent doses."
)


def trim_words(text, n):
    w = text.split()
    return " ".join(w[:n]) + (" ..." if len(w) > n else "")


def trim_tokens(text, n, tok):
    """Cut text to n tokens. Token budgets (not word counts) keep every prompt inside the length limit: numeric-heavy
    text can use twice as many tokens per word as prose."""
    ids = tok(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= n else tok.decode(ids[:n]) + " ..."


def build_user(mode, question, refs, excerpts, ref_words=300, pdf_words=180, tok=None, ref_tokens=380, pdf_tokens=200):
    """refs: list of (question, answer); excerpts: list of (label, text). With tok given, trimming is by tokens."""
    def cut(text, words, tokens):
        return trim_tokens(text, tokens, tok) if tok is not None else trim_words(text, words)
    if mode == "none":
        return "Question: " + question
    parts = ["Reference answers to similar questions:"]
    for i, (q, a) in enumerate(refs, 1):
        parts.append("[%d] Q: %s\nA: %s" % (i, q, cut(a, ref_words, ref_tokens)))
    if mode == "full" and excerpts:
        parts.append("Source excerpts:")
        for label, t in excerpts:
            parts.append("[%s] %s" % (label, cut(t, pdf_words, pdf_tokens)))
    parts.append("Question: " + question)
    return "\n\n".join(parts)


class Context:
    """Holds the lookups needed to build prompts for any query set."""

    def __init__(self, rows, page_text=None, chunk_text=None):
        self.qid2row = {r["question_id"]: r for r in rows}
        self.rows = rows
        self.page_text = page_text or {}      # (doc_id, page) -> first chunk text of that page
        self.chunk_text = chunk_text or []    # chunk id -> text
        self._cit = {}

    def citations(self, qid):
        if qid not in self._cit:
            self._cit[qid] = json.loads(self.qid2row[qid]["citations_json"])
        return self._cit[qid]

    def propagated_pages(self, neighbour_ids, limit=2):
        """Pages cited by the nearest training questions, in neighbour order. Returns [(label, text)]."""
        out, seen = [], set()
        for nid in neighbour_ids:
            for c in self.citations(nid):
                if c.get("kind") != "zoho":
                    continue
                for p in c.get("pages", []):
                    key = (c["doc_id"], p)
                    if key in self.page_text and key not in seen:
                        seen.add(key)
                        name = (c.get("name") or "source")[:60]
                        out.append(("%s, p.%d" % (name, p), self.page_text[key]))
                        if len(out) == limit:
                            return out
        return out


def make_example(ctx, mode, qid, question, neighbours, chunk_hits=None, answer=None, noise=0.0, rng=None,
                 pool_qids=None, n_refs=2, tok=None):
    """neighbours: [[question_id, score], ...] best first."""
    rng = rng or random.Random(0)
    nids = [n[0] for n in neighbours]
    refs_ids = nids[:n_refs]
    if noise and pool_qids and len(refs_ids) == n_refs and rng.random() < noise:
        refs_ids[-1] = rng.choice(pool_qids)      # RAFT-style distractor: the model learns to ignore bad context
    refs = [(ctx.qid2row[i]["question"], ctx.qid2row[i]["final_answer"]) for i in refs_ids]
    excerpts = []
    if mode == "full":
        excerpts = ctx.propagated_pages(nids[:3])
        if len(excerpts) < 2 and chunk_hits and ctx.chunk_text:
            for cid in chunk_hits:
                if len(excerpts) == 2:
                    break
                excerpts.append(("retrieved passage", ctx.chunk_text[cid]))
        if noise and excerpts and pool_qids and rng.random() < noise:
            other = ctx.propagated_pages([rng.choice(pool_qids)], limit=1)
            if other:
                excerpts[-1] = other[0]
    ex = {"qid": qid, "mode": mode, "system": SYSTEM, "user": build_user(mode, question, refs, excerpts, tok=tok)}
    if answer is not None:
        ex["answer"] = answer
    return ex

In [ ]:
%%writefile train_qlora.py
"""QLoRA fine-tuning with prompt-masked loss (only the answer tokens are trained on).

Uses the plain transformers Trainer + peft (no TRL) so it does not depend on TRL's changing API.
Run one process per GPU:  CUDA_VISIBLE_DEVICES=0 python train_qlora.py --train t.jsonl --dev d.jsonl --out dir

Safety nets for long Kaggle runs: a wall-clock limit (--train_hours) that saves the adapter and stops cleanly,
a NaN guard, checkpoints every --save_steps with automatic resume, and a tokens/sec estimate for planning.
--tiny / --no_quant exist only for CPU smoke tests. Pure ASCII source on purpose.
"""
import argparse
import inspect
import json
import math
import os
import random
import time

import torch
from transformers import (AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer,
                          TrainerCallback, TrainingArguments)
from peft import LoraConfig, get_peft_model

TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--train", required=True)
    p.add_argument("--dev", default="")
    p.add_argument("--out", required=True)
    p.add_argument("--model", default="Qwen/Qwen2.5-3B-Instruct")
    p.add_argument("--max_len", type=int, default=2560)
    p.add_argument("--epochs", type=float, default=1.0)
    p.add_argument("--max_steps", type=int, default=-1)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--bs", type=int, default=1)
    p.add_argument("--grad_accum", type=int, default=16)
    p.add_argument("--lora_r", type=int, default=16)
    p.add_argument("--lora_alpha", type=int, default=32)
    p.add_argument("--save_steps", type=int, default=100)
    p.add_argument("--eval_steps", type=int, default=100)
    p.add_argument("--train_hours", type=float, default=10.0)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--optim", default="paged_adamw_8bit")
    p.add_argument("--no_quant", action="store_true")
    p.add_argument("--tiny", action="store_true")
    p.add_argument("--no_grad_ckpt", action="store_true")
    p.add_argument("--ddp_backend", default="", help="CPU smoke tests only (gloo); GPUs use NCCL automatically")
    return p.parse_args()


def encode_example(tok, ex, max_len, im_end_id):
    """Chat-format the prompt, append the answer, and mask the prompt in the labels. None if the prompt leaves no room."""
    msgs = [{"role": "system", "content": ex["system"]}, {"role": "user", "content": ex["user"]}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    prompt = tok(text, add_special_tokens=False)["input_ids"]
    ans = tok(ex["answer"], add_special_tokens=False)["input_ids"]
    room = max_len - len(prompt)
    if room < 256:
        return None
    if len(ans) + 1 > room:
        ans = ans[:room]  # truncated: no end token, so the model is never taught to stop mid-sentence
        truncated = True
    else:
        ans = ans + [im_end_id]
        truncated = False
    return {"input_ids": prompt + ans, "labels": [-100] * len(prompt) + ans, "truncated": truncated,
            "n_prompt": len(prompt)}


def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]


class ListDataset(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return {"input_ids": self.items[i]["input_ids"], "labels": self.items[i]["labels"]}


class Collator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, feats):
        m = max(len(f["input_ids"]) for f in feats)
        ids = torch.full((len(feats), m), self.pad_id, dtype=torch.long)
        lab = torch.full((len(feats), m), -100, dtype=torch.long)
        att = torch.zeros((len(feats), m), dtype=torch.long)
        for i, f in enumerate(feats):
            n = len(f["input_ids"])
            ids[i, :n] = torch.tensor(f["input_ids"])
            lab[i, :n] = torch.tensor(f["labels"])
            att[i, :n] = 1
        return {"input_ids": ids, "labels": lab, "attention_mask": att}


def _dist():
    return torch.distributed.is_available() and torch.distributed.is_initialized()


class Guard(TrainerCallback):
    """Wall-clock limit + NaN guard. On a stop, a checkpoint is saved (so the next session resumes) and the adapter is
    written. With several processes (DDP) the stop decision is all-reduced, so every rank stops on the same step."""

    def __init__(self, hours):
        self.t_end = time.time() + hours * 3600
        self.reason = "completed"
        self.nan = False
        self.start_step = 0

    def on_train_begin(self, args, state, control, **kw):
        self.start_step = state.global_step      # > 0 when resumed from a checkpoint

    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs and (math.isnan(logs["loss"]) or math.isinf(logs["loss"])):
            self.nan = True
        return control

    def on_step_end(self, args, state, control, **kw):
        flag = 2 if self.nan else (1 if time.time() > self.t_end else 0)
        if _dist():
            t = torch.tensor([flag], device="cuda" if torch.cuda.is_available() else "cpu")
            torch.distributed.all_reduce(t, op=torch.distributed.ReduceOp.MAX)
            flag = int(t.item())
        if flag:
            self.reason = "nan_loss" if flag == 2 else "time_limit"
            if flag == 2:
                print("NaN/inf loss - stopping. Try a smaller model or lower lr.", flush=True)
            else:
                control.should_save = True               # checkpoint for the next session
            control.should_training_stop = True
        return control


def build_model(a, tok):
    cuda = torch.cuda.is_available()
    if a.tiny:
        cfg = AutoConfig.for_model("qwen2", vocab_size=len(tok), hidden_size=64, intermediate_size=128,
                                   num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=2,
                                   max_position_embeddings=4096, tie_word_embeddings=True)
        return AutoModelForCausalLM.from_config(cfg)
    if a.no_quant:
        return AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=torch.float16 if cuda else torch.float32)
    from peft import prepare_model_for_kbit_training
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    rank = int(os.environ.get("LOCAL_RANK", 0))       # one full 4-bit copy per GPU under DDP
    model = AutoModelForCausalLM.from_pretrained(a.model, quantization_config=bnb, torch_dtype=torch.float16,
                                                 device_map={"": rank}, attn_implementation="sdpa")
    return prepare_model_for_kbit_training(model, use_gradient_checkpointing=not a.no_grad_ckpt,
                                           gradient_checkpointing_kwargs={"use_reentrant": False})


def main():
    a = parse()
    random.seed(a.seed)
    torch.manual_seed(a.seed)
    os.makedirs(a.out, exist_ok=True)
    cuda = torch.cuda.is_available()
    tok = AutoTokenizer.from_pretrained(a.model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    im_end_id = tok.convert_tokens_to_ids("<|im_end|>")
    assert im_end_id is not None and im_end_id != tok.unk_token_id, "tokenizer has no <|im_end|> (use a Qwen instruct model)"

    t0 = time.time()
    train_raw = load_jsonl(a.train)
    enc = [e for e in (encode_example(tok, ex, a.max_len, im_end_id) for ex in train_raw) if e]
    avg_len = sum(len(e["input_ids"]) for e in enc) / max(1, len(enc))
    print("train examples: %d kept of %d | truncated answers: %d | avg tokens %.0f | max %d | tokenised in %.0fs" % (
        len(enc), len(train_raw), sum(e["truncated"] for e in enc), avg_len, max(len(e["input_ids"]) for e in enc),
        time.time() - t0), flush=True)
    dev_ds = None
    if a.dev:
        dev_enc = [e for e in (encode_example(tok, ex, a.max_len, im_end_id) for ex in load_jsonl(a.dev)) if e]
        dev_ds = ListDataset(dev_enc)
        print("dev examples:", len(dev_enc), flush=True)

    model = build_model(a, tok)
    model.config.use_cache = False
    if a.no_quant and not a.tiny and not a.no_grad_ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=a.lora_r, lora_alpha=a.lora_alpha, lora_dropout=0.05,
                                             target_modules=TARGETS, task_type="CAUSAL_LM"))
    for p in model.parameters():  # fp16 training needs fp32 trainable weights (GradScaler cannot unscale fp16 grads)
        if p.requires_grad:
            p.data = p.data.float()
    model.print_trainable_parameters()

    sig = inspect.signature(TrainingArguments.__init__).parameters
    kw = dict(output_dir=a.out, per_device_train_batch_size=a.bs, per_device_eval_batch_size=1,
              gradient_accumulation_steps=a.grad_accum, learning_rate=a.lr, lr_scheduler_type="cosine",
              num_train_epochs=a.epochs, max_steps=a.max_steps, logging_steps=5,
              save_steps=a.save_steps, save_total_limit=2, fp16=cuda, optim=a.optim, report_to="none",
              remove_unused_columns=False, dataloader_num_workers=0, seed=a.seed, max_grad_norm=1.0,
              disable_tqdm=True, gradient_checkpointing=(cuda and not a.no_grad_ckpt and not a.no_quant))
    if dev_ds is not None:
        kw["eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"] = "steps"
        kw["eval_steps"] = a.eval_steps
    if "gradient_checkpointing_kwargs" in sig:
        kw["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    world = int(os.environ.get("WORLD_SIZE", 1))
    if world > 1:
        kw["ddp_find_unused_parameters"] = False
    if a.ddp_backend:
        kw["ddp_backend"] = a.ddp_backend
    # 3% warm-up: old transformers has warmup_ratio; new transformers takes a float in warmup_steps as the ratio
    kw["warmup_ratio" if "warmup_ratio" in sig else "warmup_steps"] = 0.03
    dropped = sorted(k for k in kw if k not in sig)
    if dropped:
        print("WARNING: this transformers version has no TrainingArguments option(s):", dropped, "- ignoring them", flush=True)
        kw = {k: v for k, v in kw.items() if k in sig}
    guard = Guard(a.train_hours)
    trainer = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=ListDataset(enc), eval_dataset=dev_ds,
                      data_collator=Collator(tok.pad_token_id), callbacks=[guard])
    resume = any(d.startswith("checkpoint-") for d in os.listdir(a.out))
    t1 = time.time()
    trainer.train(resume_from_checkpoint=True if resume else None)
    elapsed = time.time() - t1
    final = os.path.join(a.out, "final")
    trainer.save_model(final)                      # main process only (Trainer handles it)
    if not trainer.is_world_process_zero():
        return
    tok.save_pretrained(final)
    steps = trainer.state.global_step
    run_steps = steps - guard.start_step
    per_step = elapsed / max(1, run_steps)
    ex_per_step = a.bs * a.grad_accum * world
    summary = {"status": guard.reason, "steps": steps, "steps_this_session": run_steps, "resumed_from_step": guard.start_step,
               "world_size": world, "elapsed_s": round(elapsed, 1), "sec_per_step": round(per_step, 2),
               "examples_per_step": ex_per_step, "avg_tokens": round(avg_len),
               "tokens_per_sec_est": round(avg_len * ex_per_step / max(1e-9, per_step)),
               "n_train": len(enc), "truncated_answers": sum(e["truncated"] for e in enc),
               "steps_per_epoch": len(enc) // ex_per_step, "max_len": a.max_len,
               "loss_log": [(h["step"], round(h["loss"], 4)) for h in trainer.state.log_history if "loss" in h],
               "eval_log": [(h["step"], round(h["eval_loss"], 4)) for h in trainer.state.log_history if "eval_loss" in h]}
    json.dump(summary, open(os.path.join(a.out, "train_summary.json"), "w"), indent=1)
    print("DONE", {k: v for k, v in summary.items() if k not in ("loss_log", "eval_log")}, flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile infer.py
"""Phase 5 generation worker: one process per GPU.

Reads a jobs file (jsonl: uid, adapter, prio, system, user), generates greedy answers in batches, and appends every
finished batch to --out immediately, so a crashed or time-limited run resumes where it stopped (done uids are skipped).
adapter is "nn" / "none" (a LoRA folder under --adapter_dir) or "base" (the same model with the adapter switched off:
the zero-shot baseline B1).  Pure ASCII source on purpose.

    CUDA_VISIBLE_DEVICES=0 python infer.py --jobs jobs.jsonl --adapter_dir /path --adapters nn --out gen_gpu0.jsonl
"""
import argparse
import json
import os
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
from peft import PeftModel


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--jobs", required=True)
    p.add_argument("--adapter_dir", required=True)
    p.add_argument("--adapters", required=True, help="comma list handled by this worker, e.g. none,base")
    p.add_argument("--out", required=True)
    p.add_argument("--model", default="Qwen/Qwen2.5-3B-Instruct")
    p.add_argument("--max_new", type=int, default=768)
    p.add_argument("--batch", type=int, default=12)
    p.add_argument("--rep_penalty", type=float, default=1.05, help="HF repetition penalty (acts on prompt AND answer tokens)")
    p.add_argument("--gen_rep_penalty", type=float, default=1.0, help="penalty on tokens of the ANSWER only (1.0 = off)")
    p.add_argument("--gen_no_repeat", type=int, default=0, help="forbid repeating any n-gram of the ANSWER (0 = off)")
    p.add_argument("--do_sample", action="store_true")
    p.add_argument("--temperature", type=float, default=0.7)
    p.add_argument("--top_p", type=float, default=0.9)
    p.add_argument("--seed", type=int, default=1234)
    p.add_argument("--hours", type=float, default=10.0)
    p.add_argument("--no_quant", action="store_true")
    return p.parse_args()


class GenRepetitionPenalty(LogitsProcessor):
    """Repetition penalty over the generated tokens only (the HF one also penalises every prompt token, which
    discourages copying wording from the reference answers in the prompt). Assumes left padding."""

    def __init__(self, penalty, prompt_len):
        self.p, self.plen = penalty, prompt_len

    def __call__(self, input_ids, scores):
        g = input_ids[:, self.plen:]
        if g.shape[1] == 0:
            return scores
        s = torch.gather(scores, 1, g)
        s = torch.where(s < 0, s * self.p, s / self.p)
        return scores.scatter(1, g, s)


class GenNoRepeatNGram(LogitsProcessor):
    """Forbid any n-gram that already occurs in the generated answer (prompt excluded). Incremental per row."""

    def __init__(self, n, prompt_len):
        self.n, self.plen, self.seen = n, prompt_len, None

    def __call__(self, input_ids, scores):
        B, L = input_ids.shape
        if self.seen is None:
            self.seen = [dict() for _ in range(B)]
        glen = L - self.plen
        if glen < self.n - 1:
            return scores
        rows = input_ids[:, self.plen:].tolist()
        for b in range(B):
            g = rows[b]
            if glen >= self.n:                      # register the newest complete n-gram
                self.seen[b].setdefault(tuple(g[-self.n:-1]), set()).add(g[-1])
            banned = self.seen[b].get(tuple(g[len(g) - self.n + 1:]))
            if banned:
                scores[b, list(banned)] = -float("inf")
        return scores


def load_model(a, names):
    cuda = torch.cuda.is_available()
    if a.no_quant:
        base = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=torch.float16 if cuda else torch.float32)
        if cuda:
            base = base.to("cuda:0")
    else:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True)
        base = AutoModelForCausalLM.from_pretrained(a.model, quantization_config=bnb, torch_dtype=torch.float16,
                                                    device_map={"": 0})
    real = [n for n in names if n != "base"]
    if not real:
        return base.eval(), False
    model = PeftModel.from_pretrained(base, os.path.join(a.adapter_dir, real[0]), adapter_name=real[0])
    for n in real[1:]:
        model.load_adapter(os.path.join(a.adapter_dir, n), adapter_name=n)
    return model.eval(), True


def generate(model, tok, texts, a, device):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
    plen = enc["input_ids"].shape[1]
    procs = LogitsProcessorList()
    if getattr(a, "gen_rep_penalty", 1.0) != 1.0:
        procs.append(GenRepetitionPenalty(a.gen_rep_penalty, plen))
    if getattr(a, "gen_no_repeat", 0):
        procs.append(GenNoRepeatNGram(a.gen_no_repeat, plen))
    kw = dict(max_new_tokens=a.max_new, repetition_penalty=a.rep_penalty, pad_token_id=tok.pad_token_id,
              logits_processor=procs)
    if getattr(a, "do_sample", False):
        torch.manual_seed(a.seed)
        kw.update(do_sample=True, temperature=a.temperature, top_p=a.top_p, top_k=0)
    else:
        kw.update(do_sample=False, temperature=None, top_p=None, top_k=None)
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    gen = out[:, enc["input_ids"].shape[1]:]
    res = []
    for g in gen:
        ids = g.tolist()
        n = len(ids)
        while n and ids[n - 1] == tok.pad_token_id:
            n -= 1
        res.append((tok.decode(ids[:n], skip_special_tokens=True).strip(), n))
    return res


def main():
    a = parse()
    names = a.adapters.split(",")
    jobs = [json.loads(l) for l in open(a.jobs, encoding="utf-8")]
    jobs = [j for j in jobs if j["adapter"] in names]
    done = set()
    if os.path.exists(a.out):
        done = {json.loads(l)["uid"] for l in open(a.out, encoding="utf-8") if l.strip()}
    todo = [j for j in jobs if j["uid"] not in done]
    print("jobs for %s: %d | already done: %d | to do: %d" % (names, len(jobs), len(done), len(todo)), flush=True)
    if not todo:
        return
    tok = AutoTokenizer.from_pretrained(a.model)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    for j in todo:
        j["text"] = tok.apply_chat_template([{"role": "system", "content": j["system"]}, {"role": "user", "content": j["user"]}],
                                            tokenize=False, add_generation_prompt=True)
        j["plen"] = len(tok(j["text"], add_special_tokens=False)["input_ids"])
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model, has_peft = load_model(a, names)
    t_end = time.time() + a.hours * 3600
    t0 = time.time()
    n_done = 0
    fout = open(a.out, "a", encoding="utf-8")
    for name in names:
        group = sorted((j for j in todo if j["adapter"] == name), key=lambda j: (j["prio"], -j["plen"]))
        if not group:
            continue
        if name != "base" and has_peft:
            model.set_adapter(name)
        print("adapter %s: %d jobs" % (name, len(group)), flush=True)
        i = 0
        bs = a.batch
        while i < len(group):
            if time.time() > t_end:
                print("time limit reached - stopping; rerun to resume", flush=True)
                return
            batch = group[i:i + bs]
            try:
                if name == "base" and has_peft:
                    with model.disable_adapter():
                        res = generate(model, tok, [j["text"] for j in batch], a, device)
                else:
                    res = generate(model, tok, [j["text"] for j in batch], a, device)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                print("out of memory - batch size now", bs, flush=True)
                continue
            for j, (text, n) in zip(batch, res):
                fout.write(json.dumps({"uid": j["uid"], "adapter": name, "text": text, "n_tokens": n,
                                       "hit_limit": n >= a.max_new}, ensure_ascii=False) + "\n")
            fout.flush()
            i += len(batch)
            n_done += len(batch)
            el = time.time() - t0
            print("[%5.0f min] %s %d/%d | %.1f s per answer" % (el / 60, name, i, len(group), el / n_done), flush=True)
    print("DONE", n_done, "answers in %.1f min" % ((time.time() - t0) / 60), flush=True)


if __name__ == "__main__":
    main()

## 0. Configuration and resume detection

In [ ]:
import glob, os, sys, json, gzip, csv, random, time, shutil, subprocess, collections as C
csv.field_size_limit(10 ** 9)
_missing = [f for f in ("prompts.py", "train_qlora.py", "infer.py") if not os.path.exists(f)]
assert not _missing, "missing " + str(_missing) + " - use Run All from the top"
sys.path.insert(0, os.getcwd())
import torch
EXTRA = os.environ.get("P8_EXTRA", "").split()        # CPU smoke-test hook only; empty on Kaggle
if not EXTRA:
    assert torch.cuda.is_available(), "No GPU. Session options -> Accelerator -> GPU T4 x2, then Run All."
N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else int(os.environ.get("P8_NPROC", "1"))
print("GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

MODEL = os.environ.get("P8_MODEL", "Qwen/Qwen2.5-3B-Instruct")
MODE = "nn"
N_TRAIN = int(os.environ.get("P8_NTRAIN", "0"))   # 0 = all 12,825 training rows
EPOCHS = 1.0
MAX_LEN = 3072
EFFECTIVE_BATCH = 16
NOISE = 0.2
N_DEV = 60
TRAIN_HOURS = float(os.environ.get("P8_HOURS", "10.5"))   # leaves ~1.5 h of the 12 h session for setup and the final cells
SAVE_STEPS = 25                                          # ~20 min of training between checkpoints
WORK = os.environ.get("P8_WORK", "/kaggle/working")
INPUT = os.environ.get("P8_INPUT", "/kaggle/input")
OUT = WORK + "/adapters/" + MODE

def find(pattern, required=True):
    hit = sorted(glob.glob(INPUT + "/**/" + pattern, recursive=True))
    if not hit and required:
        raise AssertionError("attach a dataset/notebook output containing " + pattern)
    return hit[0] if hit else None

def op(path):
    return gzip.open(path, "rt", encoding="utf-8") if path.endswith(".gz") else open(path, encoding="utf-8")

TRAIN_CSV = find("train_processed*.csv")
NN_PATH = find("question_neighbours_reranked.json*")
os.makedirs(WORK + "/data", exist_ok=True)

# resume: the latest checkpoint and the exact training prompts of a previous session of this notebook
import re
prev = [p.replace("\\", "/") for p in glob.glob(INPUT + "/**/adapters/%s/checkpoint-*/trainer_state.json" % MODE, recursive=True)]
prev.sort(key=lambda p: int(re.search(r"checkpoint-(\d+)", p).group(1)))
RESUMED_FROM = None
if prev and not os.path.exists(OUT):
    ck = os.path.dirname(prev[-1])
    RESUMED_FROM = os.path.basename(ck)
    shutil.copytree(ck, os.path.join(OUT, RESUMED_FROM))
    root = ck.rsplit("/adapters/", 1)[0]
    for f in ("train_%s.jsonl" % MODE, "dev_%s.jsonl" % MODE):
        p = os.path.join(root, "data", f)
        assert os.path.exists(p), "previous output has a checkpoint but no data/%s - cannot resume safely" % f
        shutil.copy(p, os.path.join(WORK, "data", f))
    for d in glob.glob(root + "/logs/*"):
        os.makedirs(WORK + "/logs", exist_ok=True); shutil.copy(d, WORK + "/logs/prev_" + os.path.basename(d))
print("train csv:", TRAIN_CSV, "| neighbours:", NN_PATH)
print("RESUMING from", RESUMED_FROM if RESUMED_FROM else "- (fresh run)")

## 1. Build the training prompts (skipped when resuming: the previous session's prompts are reused byte for byte)

In [ ]:
from transformers import AutoTokenizer
import prompts as P
tok = AutoTokenizer.from_pretrained(MODEL)
rows = list(csv.DictReader(open(TRAIN_CSV, encoding="utf-8")))
TR, DV = "%s/data/train_%s.jsonl" % (WORK, MODE), "%s/data/dev_%s.jsonl" % (WORK, MODE)
if not RESUMED_FROM:
    NN = json.load(op(NN_PATH))
    ctx = P.Context(rows)
    train_rows = [r for r in rows if r["split"] == "train"]
    dev_rows = [r for r in rows if r["split"] == "dev"][:N_DEV]
    sub = train_rows if not N_TRAIN else random.Random(42).sample(train_rows, min(N_TRAIN, len(train_rows)))
    pool = [r["question_id"] for r in train_rows]
    missing = [r["question_id"] for r in sub if r["question_id"] not in NN["train"]]
    assert not missing, "%d training rows have no re-ranked neighbours" % len(missing)
    for path, items, split, noise in ((TR, sub, "train", NOISE), (DV, dev_rows, "dev", 0.0)):
        r = random.Random(1)
        with open(path, "w", encoding="utf-8") as f:
            for row in items:
                q = row["question_id"]
                ex = P.make_example(ctx, MODE, q, row["question"], NN[split][q], answer=row["final_answer"], noise=noise,
                                    rng=r, pool_qids=pool, tok=tok)
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")
ex = [json.loads(l) for l in open(TR, encoding="utf-8")]
lens = [len(tok(e["system"] + e["user"], add_special_tokens=False)["input_ids"]) for e in ex[:500]]
print("training examples: %d | dev: %d | prompt tokens (first 500): mean %.0f max %d" % (
    len(ex), sum(1 for _ in open(DV, encoding="utf-8")), sum(lens) / len(lens), max(lens)))
print("\n--- example prompt ---\n%s\n--- target starts: %s" % (ex[0]["user"][:1500], ex[0]["answer"][:200].replace("\n", " ")))

## 2. Pilot on the 8 LONGEST examples: memory at MAX_LEN and speed, two GPUs (falls back to one if DDP fails)

In [ ]:
if not EXTRA:
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL)

def tl(e):
    return len(tok(e["system"] + e["user"] + e["answer"], add_special_tokens=False)["input_ids"])
longest = sorted(ex, key=tl, reverse=True)[:8]
with open(WORK + "/data/pilot.jsonl", "w", encoding="utf-8") as f:
    for e in longest:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
print("pilot examples: %d-%d tokens (cut to MAX_LEN=%d)" % (tl(longest[-1]), tl(longest[0]), MAX_LEN))

ENV = dict(os.environ, TOKENIZERS_PARALLELISM="false", NCCL_P2P_DISABLE="1", NCCL_IB_DISABLE="1", USE_LIBUV="0")

def launch(nproc, args, log=None):
    base = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", str(nproc)] if nproc > 1 else [sys.executable]
    env = dict(ENV, CUDA_VISIBLE_DEVICES=",".join(str(i) for i in range(max(1, nproc)))) if not EXTRA else ENV
    if log is None:
        return subprocess.run(base + ["train_qlora.py"] + args + EXTRA, env=env, capture_output=True, text=True)
    return subprocess.Popen(base + ["train_qlora.py"] + args + EXTRA, env=env, stdout=log, stderr=subprocess.STDOUT)

def targs(train, out, accum, extra=()):
    return ["--train", train, "--dev", DV, "--out", out, "--model", MODEL, "--max_len", str(MAX_LEN), "--grad_accum", str(accum),
            "--epochs", str(EPOCHS), "--train_hours", str(TRAIN_HOURS), "--save_steps", str(SAVE_STEPS), "--eval_steps", "100"] + list(extra)

NPROC = max(1, N_GPU)
for attempt in ([NPROC, 1] if NPROC > 1 else [1]):
    shutil.rmtree(WORK + "/pilot", ignore_errors=True)
    r = launch(attempt, targs(WORK + "/data/pilot.jsonl", WORK + "/pilot", 2, ["--max_steps", "2", "--save_steps", "1000", "--eval_steps", "1000"]))
    if r.returncode == 0:
        NPROC = attempt
        break
    print("pilot with %d process(es) FAILED:\n" % attempt, r.stdout[-2500:], r.stderr[-2500:])
else:
    raise AssertionError("pilot failed on 1 GPU too - read the error above (out of memory: lower MAX_LEN to 2560)")
s = json.load(open(WORK + "/pilot/train_summary.json"))
GRAD_ACCUM = EFFECTIVE_BATCH // NPROC
per_ex = s["sec_per_step"] / s["examples_per_step"] * NPROC        # seconds per example on one GPU (longest examples)
done = 0
if RESUMED_FROM:
    done = int(RESUMED_FROM.split("-")[1]) * EFFECTIVE_BATCH
proj = per_ex * (len(ex) * EPOCHS - done) / NPROC / 3600
print("pilot OK with %d GPU process(es) | %.1f s per (longest) example per GPU | effective batch %d = %d x %d accumulation x %d GPU" % (
    NPROC, per_ex, EFFECTIVE_BATCH, 1, GRAD_ACCUM, NPROC))
print("upper-bound projection for the remaining %d examples: %.1f h (real examples are shorter; TRAIN_HOURS=%.1f)" % (
    len(ex) * EPOCHS - done, proj, TRAIN_HOURS))
if torch.cuda.is_available():
    print("peak GPU memory in the pilot is in its log; the run will use MAX_LEN=%d" % MAX_LEN)
shutil.rmtree(WORK + "/pilot", ignore_errors=True)

## 3. Train (checkpoint every 25 steps; resumes automatically)

In [ ]:
os.makedirs(WORK + "/logs", exist_ok=True)
log = open("%s/logs/train_%s.log" % (WORK, MODE), "a")
p = launch(NPROC, targs(TR, OUT, GRAD_ACCUM), log=log)
t0 = time.time()
while p.poll() is None:
    time.sleep(180 if not EXTRA else 5)
    lines = [l for l in open("%s/logs/train_%s.log" % (WORK, MODE), errors="ignore").read().splitlines()
             if any(k in l for k in ("'loss'", "eval_loss", "DONE", "Error", "Traceback", "examples", "trainable"))]
    print("[%5.0f min] %s" % ((time.time() - t0) / 60, lines[-1][:200] if lines else "(starting)"), flush=True)
print("training process exit code", p.returncode)
if p.returncode:
    print(open("%s/logs/train_%s.log" % (WORK, MODE), errors="ignore").read()[-4000:])

## 4. Result: finished or needs another session?

In [ ]:
s = json.load(open(OUT + "/train_summary.json"))
print({k: v for k, v in s.items() if k not in ("loss_log", "eval_log")})
print("train loss (every ~50 steps):", s["loss_log"][::10])
print("dev loss:", s["eval_log"])
total = s["steps_per_epoch"] * EPOCHS
print("\nSTATUS: %s | step %d of ~%d" % (s["status"], s["steps"], total))
if s["status"] == "time_limit":
    print(">>> NOT FINISHED. Run this notebook again with THIS notebook's output attached (Add Input -> Your Work); it resumes from the last checkpoint.")
elif s["status"] == "completed":
    print(">>> FINISHED. The adapter is in adapters/%s/final. Attach this notebook's output to the inference notebook." % MODE)

## 5. Sanity check: 3 dev answers with the fixed decoding (answer-only penalty 1.1 + no repeated 12-token n-gram)

In [ ]:
if EXTRA or s["status"] != "completed":
    print("skipped (smoke test or unfinished run)")
else:
    import infer
    from types import SimpleNamespace
    ga = SimpleNamespace(no_quant=False, model=MODEL, adapter_dir=WORK + "/adapters", max_new=1280, rep_penalty=1.0, gen_rep_penalty=1.1,
                         gen_no_repeat=12, do_sample=False, temperature=0.7, top_p=0.9, seed=0)
    model, _ = infer.load_model(SimpleNamespace(no_quant=False, model=MODEL, adapter_dir=OUT), ["final"])
    tok.padding_side = "left"
    dev = [json.loads(l) for l in open(DV, encoding="utf-8")][:3]
    texts = [tok.apply_chat_template([{"role": "system", "content": e["system"]}, {"role": "user", "content": e["user"]}],
                                     tokenize=False, add_generation_prompt=True) for e in dev]
    for e, (t, n) in zip(dev, infer.generate(model, tok, texts, ga, "cuda:0")):
        print("=" * 90, "\nQ:", e["user"].split("Question: ")[-1][:160], "\n", t[:900], "\n... (%d tokens, %d words)" % (n, len(t.split())))

## After it finishes
* **Section 4 says NOT FINISHED:** open this notebook -> **Add Input -> Your Work -> this notebook** (its latest version) -> **Save & Run All** again.
  It copies the last checkpoint and the exact training prompts from that output and continues.
* **Section 4 says FINISHED:** send me the notebook link. The adapter is `adapters/nn/final`; the evaluation/inference notebook attaches this output directly.